In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path('../004 data')

## 1. Load Data

> **Forutsetning:** Kjør `data_prep.ipynb` først for å generere `costs_clean.csv` og `specs_clean.csv`.

### 1.1 Yacht Specifications

In [ ]:
# Load cleaned yacht specs
specs = pd.read_csv(DATA_DIR / 'specs_clean.csv')

# Normalize yacht IDs
specs['yacht_id']      = specs['Yacht'].str.strip().str.lower()
specs['yacht_id_base'] = specs['yacht_id'].str.replace(r'-\\d+$', '', regex=True)

# Ensure numeric types (already clean, but safe to recast)
for col in ['GT', 'NT', 'LOA (m)', 'Reg. Length (m)', 'Beam (m)', 'Draft (m)',
            'Air Draft (m)', 'Fuel consumption (L/h)']:
    specs[col] = pd.to_numeric(specs[col], errors='coerce')

print(f'Yacht specs: {len(specs)} entries')
print(f'Størrelses-kategorier: {specs[\"Størrelses-kategori\"].value_counts().to_dict()}')
print(f'Med loskrav (LOA>70m): {(specs[\"Loskrav (LOA>70m)\"] == \"Ja\").sum()} yachter')
specs

### 1.2 Cost Data (costs_clean.csv)

Ferdig renset og parset av `data_prep.ipynb`. Inneholder alle transaksjoner 2020–2025 med typede kolonner og `flag`-kolonne for datakvalitetsproblemer.

In [ ]:
# Load cleaned cost data
costs_all = pd.read_csv(DATA_DIR / 'costs_clean.csv', parse_dates=['arrival_date', 'departure_date'])

# Restore categoricals
costs_all['office']       = costs_all['office'].astype('category')
costs_all['service_type'] = costs_all['service_type'].astype('category')

print(f'Total transaction rows: {len(costs_all)}')
print(f'\nTransactions per year (all data):')
print(costs_all['year'].value_counts().sort_index())
print(f'\nFlagged rows: {(costs_all[\"flag\"].notna() & (costs_all[\"flag\"] != \"\")).sum()}')
costs_all.head()

In [ ]:
# Filter to 2020–2024 data (analysis scope)
costs = costs_all[costs_all['year'].between(2020, 2024)].copy().reset_index(drop=True)
print(f'Transactions 2020–2024: {len(costs)}')
print(f'\nTransactions per year:')
print(costs['year'].value_counts().sort_index())
print(f'\nYachts in 2020–2024 data: {sorted(costs["yacht_id"].unique())}')
costs.head()

### 1.3 Merge Costs with Yacht Specs

In [ ]:
# Create a specs lookup by base yacht_id (average numeric specs for yacht_2 variants)
agg_cols = ['GT', 'NT', 'LOA (m)', 'Reg. Length (m)', 'Beam (m)', 'Draft (m)', 'Air Draft (m)', 'Fuel consumption (L/h)']
specs_base = specs.groupby('yacht_id_base')[agg_cols].mean().reset_index()
specs_base.rename(columns={'yacht_id_base': 'yacht_id'}, inplace=True)

# Add categorical cols (take first value per base yacht)
cat_cols = ['Loskrav (LOA>70m)', 'Størrelses-kategori']
specs_cat = specs.groupby('yacht_id_base')[cat_cols].first().reset_index()
specs_cat.rename(columns={'yacht_id_base': 'yacht_id'}, inplace=True)
specs_base = specs_base.merge(specs_cat, on='yacht_id', how='left')

# Merge with costs
costs = costs.merge(specs_base, on='yacht_id', how='left')

matched = costs['GT'].notna().sum()
print(f'Transactions matched with specs: {matched}/{len(costs)} ({matched/len(costs)*100:.1f}%)')
costs.head()

### 1.4 Cockpit Data (cockpit_clean.csv)

In [ ]:
# Load cleaned cockpit KPIs
cockpit = pd.read_csv(DATA_DIR / 'cockpit_clean.csv', index_col='year')
cockpit.index = cockpit.index.astype(int)
cockpit = cockpit.sort_index()

print(f'Cockpit data: {len(cockpit)} years, {len(cockpit.columns)} KPI columns')
display_cols = [c for c in cockpit.columns
                if c.startswith('revenue_') or c in ('unique_boats', 'port_calls', 'days_in_network')]
cockpit[display_cols]

---
## 2. Descriptive Statistics (2020–2024)

In [ ]:
# Summary statistics for numeric columns
costs[['cost_no_vat', 'final_charge', 'margin', 'stay_days']].describe().round(2)

In [ ]:
# Transactions per yacht (2020–2024)
print('Transactions per yacht:')
print(costs['yacht_id'].value_counts().sort_index())
print(f'\nTransactions per service type:')
print(costs['service_type'].value_counts())

In [ ]:
# Transactions per office
print('Transactions per office:')
print(costs['office'].value_counts())

print(f'\nMissing values:')
print(costs.isnull().sum())

In [ ]:
# Yacht specs overview — inkluderer nye kolonner
specs[['Yacht', 'GT', 'NT', 'LOA (m)', 'Beam (m)', 'Draft (m)', 'Air Draft (m)',
       'Fuel consumption (L/h)', 'Loskrav (LOA>70m)', 'Størrelses-kategori']].describe(include='all').round(2)

---
## 3. Visualizations

### 3.1 Yacht Fleet Profile

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# GT distribution
axes[0, 0].barh(specs['Yacht'], specs['GT'], color='steelblue')
axes[0, 0].set_xlabel('Gross Tonnage (GT)')
axes[0, 0].set_title('Yacht Sizes by GT')
axes[0, 0].invert_yaxis()

# LOA vs GT
axes[0, 1].scatter(specs['LOA (m)'], specs['GT'], s=80, c='steelblue', edgecolors='black', alpha=0.7)
for _, row in specs.iterrows():
    axes[0, 1].annotate(row['Yacht'], (row['LOA (m)'], row['GT']), fontsize=7, ha='left', va='bottom')
axes[0, 1].set_xlabel('Length Overall (m)')
axes[0, 1].set_ylabel('Gross Tonnage')
axes[0, 1].set_title('LOA vs GT')

# Beam vs Draft
axes[0, 2].scatter(specs['Beam (m)'], specs['Draft (m)'], s=specs['GT']/10 + 20, c='steelblue', edgecolors='black', alpha=0.7)
for _, row in specs.iterrows():
    axes[0, 2].annotate(row['Yacht'], (row['Beam (m)'], row['Draft (m)']), fontsize=7, ha='left', va='bottom')
axes[0, 2].set_xlabel('Beam (m)')
axes[0, 2].set_ylabel('Draft (m)')
axes[0, 2].set_title('Beam vs Draft (bubble size = GT)')

# Fuel consumption
fuel = specs.dropna(subset=['Fuel consumption (L/h)'])
axes[1, 0].barh(fuel['Yacht'], fuel['Fuel consumption (L/h)'], color='coral')
axes[1, 0].set_xlabel('Fuel Consumption (L/h)')
axes[1, 0].set_title('Fuel Consumption per Yacht')
axes[1, 0].invert_yaxis()

# Størrelses-kategori distribution
kat_counts = specs['Størrelses-kategori'].value_counts()
axes[1, 1].bar(kat_counts.index, kat_counts.values, color=['steelblue', 'coral', 'seagreen'])
axes[1, 1].set_xlabel('Kategori')
axes[1, 1].set_ylabel('Antall yachter')
axes[1, 1].set_title('Flåte etter Størrelses-kategori')

# Fuel consumption vs GT
axes[1, 2].scatter(specs['GT'], specs['Fuel consumption (L/h)'], s=80, c='coral', edgecolors='black', alpha=0.7)
for _, row in specs.dropna(subset=['Fuel consumption (L/h)']).iterrows():
    axes[1, 2].annotate(row['Yacht'], (row['GT'], row['Fuel consumption (L/h)']), fontsize=7, ha='left', va='bottom')
axes[1, 2].set_xlabel('Gross Tonnage (GT)')
axes[1, 2].set_ylabel('Fuel Consumption (L/h)')
axes[1, 2].set_title('GT vs Fuel Consumption')

plt.tight_layout()
plt.show()

### 3.2 Cost Analysis (2020–2024)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Total cost per yacht (2020–2024)
yacht_cost = costs.groupby('yacht_id')['final_charge'].sum().sort_values(ascending=True)
yacht_cost.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Total Final Charge (EUR)')
axes[0].set_title('Total Cost per Yacht (2020–2024)')

# Cost per service type
svc_cost = costs.groupby('service_type')['final_charge'].sum().sort_values(ascending=True)
svc_cost.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Total Final Charge (EUR)')
axes[1].set_title('Total Cost per Service Type (2020–2024)')

# Total cost per year
year_cost = costs.groupby('year')['final_charge'].sum()
year_cost.plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Total Final Charge (EUR)')
axes[2].set_title('Total Cost per Year (2020–2024)')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Cost by year and service type (stacked bar)
pivot_svc = costs.groupby(['year', 'service_type'])['final_charge'].sum().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pivot_svc.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab10')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Final Charge (EUR)')
axes[0].set_title('Cost per Year by Service Type (2020–2024)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(fontsize=8, loc='upper left')

# Monthly distribution across all years
costs['month'] = costs['arrival_date'].dt.month
monthly = costs.groupby('month')['final_charge'].sum()
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1].bar(monthly.index, monthly.values, color='steelblue')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_labels)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Total Final Charge (EUR)')
axes[1].set_title('Monthly Cost Distribution (2020–2024 combined)')

plt.tight_layout()
plt.show()

### 3.3 Cost vs Yacht Size

In [ ]:
# Aggregate total cost per yacht, then join with specs
yacht_totals = costs.groupby('yacht_id').agg(
    total_charge=('final_charge', 'sum'),
    total_cost=('cost_no_vat', 'sum'),
    total_margin=('margin', 'sum'),
    n_transactions=('final_charge', 'count'),
    avg_stay=('stay_days', 'mean'),
    GT=('GT', 'first'),
    LOA=('LOA (m)', 'first'),
    fuel=('Fuel consumption (L/h)', 'first'),
    kategori=('Størrelses-kategori', 'first'),
    loskrav=('Loskrav (LOA>70m)', 'first'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# GT vs Total Charge, coloured by kategori
kat_colors = {'Liten': 'steelblue', 'Mellomstor': 'coral', 'Stor': 'seagreen'}
for kat, grp in yacht_totals.groupby('kategori'):
    axes[0].scatter(grp['GT'], grp['total_charge'], s=100, label=kat, color=kat_colors.get(kat, 'grey'), edgecolors='black')
for _, r in yacht_totals.iterrows():
    axes[0].annotate(r['yacht_id'], (r['GT'], r['total_charge']), fontsize=7, ha='left', va='bottom')
axes[0].set_xlabel('Gross Tonnage (GT)')
axes[0].set_ylabel('Total Final Charge (EUR)')
axes[0].set_title('GT vs Total Cost (2020–2024)')
axes[0].legend(title='Kategori')

# LOA vs Total Charge
for kat, grp in yacht_totals.groupby('kategori'):
    axes[1].scatter(grp['LOA'], grp['total_charge'], s=100, label=kat, color=kat_colors.get(kat, 'grey'), edgecolors='black')
for _, r in yacht_totals.iterrows():
    axes[1].annotate(r['yacht_id'], (r['LOA'], r['total_charge']), fontsize=7, ha='left', va='bottom')
axes[1].set_xlabel('Length Overall (m)')
axes[1].set_ylabel('Total Final Charge (EUR)')
axes[1].set_title('LOA vs Total Cost (2020–2024)')
axes[1].legend(title='Kategori')

# Fuel consumption vs Total Charge
fuel_sub = yacht_totals.dropna(subset=['fuel'])
for kat, grp in fuel_sub.groupby('kategori'):
    axes[2].scatter(grp['fuel'], grp['total_charge'], s=100, label=kat, color=kat_colors.get(kat, 'grey'), edgecolors='black')
for _, r in fuel_sub.iterrows():
    axes[2].annotate(r['yacht_id'], (r['fuel'], r['total_charge']), fontsize=7, ha='left', va='bottom')
axes[2].set_xlabel('Fuel Consumption (L/h)')
axes[2].set_ylabel('Total Final Charge (EUR)')
axes[2].set_title('Fuel Consumption vs Total Cost (2020–2024)')
axes[2].legend(title='Kategori')

plt.tight_layout()
plt.show()

### 3.4 Margin Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Margin per service type
margin_svc = costs.groupby('service_type')['margin'].sum().sort_values(ascending=True)
colors = ['red' if v < 0 else 'seagreen' for v in margin_svc]
margin_svc.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_xlabel('Total Margin (EUR)')
axes[0].set_title('Margin per Service Type (2020–2024)')
axes[0].axvline(x=0, color='black', linewidth=0.5)

# Margin per yacht
margin_yacht = costs.groupby('yacht_id')['margin'].sum().sort_values(ascending=True)
colors_y = ['red' if v < 0 else 'seagreen' for v in margin_yacht]
margin_yacht.plot(kind='barh', ax=axes[1], color=colors_y)
axes[1].set_xlabel('Total Margin (EUR)')
axes[1].set_title('Margin per Yacht (2020–2024)')
axes[1].axvline(x=0, color='black', linewidth=0.5)

# Margin per year
margin_year = costs.groupby('year')['margin'].sum()
colors_yr = ['red' if v < 0 else 'seagreen' for v in margin_year]
margin_year.plot(kind='bar', ax=axes[2], color=colors_yr)
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Total Margin (EUR)')
axes[2].set_title('Margin per Year (2020–2024)')
axes[2].axhline(y=0, color='black', linewidth=0.5)
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 3.5 Stay Duration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stay duration distribution
costs['stay_days'].dropna().hist(bins=30, ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Stay Duration (days)')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('Distribution of Stay Duration (2020–2024)')

# Stay duration vs cost
axes[1].scatter(costs['stay_days'], costs['final_charge'], alpha=0.5, c='steelblue', edgecolors='black', s=40)
axes[1].set_xlabel('Stay Duration (days)')
axes[1].set_ylabel('Final Charge (EUR)')
axes[1].set_title('Stay Duration vs Cost per Transaction (2020–2024)')

plt.tight_layout()
plt.show()

### 3.6 Cockpit Trends (2020–2025)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Revenue by service type (excluding Total)
rev_cols = [c for c in cockpit.columns if c.startswith('revenue_') and not c.endswith('_Total')]
if rev_cols:
    cockpit[rev_cols].rename(columns=lambda c: c.replace('revenue_', '')).plot(ax=axes[0, 0], marker='o')
    axes[0, 0].set_title('Revenue by Service Type (2020–2025)')
    axes[0, 0].set_ylabel('EUR (thousands)')
    axes[0, 0].legend(fontsize=8)

# Total revenue vs gross margin
rev_total = 'revenue_Total'
gm_total  = 'gm_Total'
if rev_total in cockpit.columns and gm_total in cockpit.columns:
    cockpit[[rev_total, gm_total]].rename(
        columns={rev_total: 'Revenue', gm_total: 'Gross Margin'}
    ).plot(ax=axes[0, 1], marker='o', color=['steelblue', 'seagreen'])
    axes[0, 1].set_title('Total Revenue vs Gross Margin (2020–2025)')
    axes[0, 1].set_ylabel('EUR (thousands)')

# Network activity
net_cols = [c for c in ('unique_boats', 'port_calls') if c in cockpit.columns]
if net_cols:
    cockpit[net_cols].rename(columns={'unique_boats': 'Unique Boats', 'port_calls': 'Port Calls'}).plot(
        ax=axes[1, 0], marker='o', color=['steelblue', 'coral'])
    axes[1, 0].set_title('Network Activity (2020–2025)')
    axes[1, 0].set_ylabel('Count')

# Days in network
if 'days_in_network' in cockpit.columns:
    cockpit['days_in_network'].dropna().plot(ax=axes[1, 1], marker='o', color='steelblue')
    axes[1, 1].set_title('Total Days in Network (2020–2025)')
    axes[1, 1].set_ylabel('Days')

plt.tight_layout()
plt.show()

---
## 4. Correlation Analysis

In [ ]:
# Correlation heatmap: yacht-level aggregates vs specs
corr_data = yacht_totals[['total_charge', 'total_cost', 'total_margin', 'n_transactions', 'avg_stay', 'GT', 'LOA', 'fuel']].dropna()

fig, ax = plt.subplots(figsize=(9, 7))
corr_matrix = corr_data.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Correlation Matrix: Yacht Costs vs Specifications (2020–2024)')
plt.tight_layout()
plt.show()

print('\nKey correlations with total_charge:')
print(corr_matrix['total_charge'].sort_values(ascending=False).round(3))

---
## 5. Summary of Findings

Key observations from the EDA will be filled in after running the notebook. Areas to investigate:

- **Fleet composition**: Range of yacht sizes (GT, LOA), størrelses-kategorier (Liten/Mellomstor/Stor), og loskrav for yachter over 70m
- **Drivstofforbruk**: Sammenheng mellom GT og fuel consumption (L/h) — grunnlag for fremtidige bunkering-kostnadsestimater
- **Cost drivers**: Which service types dominate costs? Which yachts are most expensive to service over 2020–2024?
- **Margin structure**: Where does the company earn its margin? Are there negative-margin services or years?
- **Seasonality**: Is there a seasonal pattern in costs across the 2020–2024 period?
- **Size–cost relationship**: How strongly does yacht size (GT/LOA/fuel consumption) predict total costs?
- **Year-over-year trends**: How have costs and margins evolved from 2020 to 2024?
- **Data gaps**: Missing fuel consumption or specs for some yachts may limit modelling accuracy

# Exploratory Data Analysis — NautiCost Dataset

This notebook performs an EDA of the NautiCost dataset, covering **2020–2024 transaction data** linked to the 17 yachts with known specifications (inkl. drivstofforbruk, loskrav og størrelses-kategori).